# 🧬 02 · Feature Engineering

> **Chapter 2.** What features ended up in the model, grouped by
> information source, with a distribution plot per group.

The trained XGBoost baseline uses **33 numeric + categorical
features** at release time (Phase 2 step 1 added 5 more gene-level
features plus an imputation flag, bringing it to 39). Each feature
comes from one of four sources:

| Group | # features | Source | Role |
|---|---:|---|---|
| Conservation | 6 | dbNSFP (phyloP, phastCons, GERP++) | "Is this site conserved across species?" |
| AA properties | 11 | Derived in `src/dbnsfp_extraction.py` | "Is the substitution chemically disruptive?" |
| Population frequency | 4 | gnomAD v2.1 + v4 | "Is this variant rare enough to be plausibly pathogenic?" |
| Gene constraint (Phase 2.1) | 6 | gnomAD v2.1.1 constraint | "Is this gene intolerant to damage in general?" |
| Imputation flags | 5 | derived | "Was this value imputed from a median rather than observed?" |
| Categorical (ref_aa / alt_aa) | 2 | derived | "Which specific AA substitution?" |

---

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path as _Path
sys.path.insert(0, str(_Path.cwd().parent))
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

REPO = Path.cwd().parent
train = pd.read_parquet(REPO / "data/splits/train.parquet")
feature_manifest = pd.read_csv(REPO / "results/metrics/xgboost_feature_columns.csv")
print(f"Encoded feature count: {len(feature_manifest)}")
print(feature_manifest.head(10).to_string(index=False))

## Conservation features — phyloP / phastCons / GERP++

All three measure *how much this site resists change across species*.
High conservation at a position where a variant appears increases the
prior that the variant is functional (and potentially damaging).

In [ ]:
conservation_cols = [
    "phyloP100way_vertebrate", "phyloP30way_mammalian",
    "phastCons100way_vertebrate", "phastCons30way_mammalian",
    "GERP++_RS", "GERP++_NR",
]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, conservation_cols):
    for lbl, color in [(0, "#2ecc71"), (1, "#e74c3c")]:
        ax.hist(train.loc[train["label"] == lbl, col].dropna(),
                bins=40, alpha=0.55, color=color, label="benign" if lbl == 0 else "pathogenic")
    ax.set_title(col, fontweight="bold")
    ax.set_yscale("log")
axes[0, 0].legend()
fig.suptitle("Conservation features — training set, pathogenic vs benign",
             y=1.02, fontweight="bold")
fig.tight_layout()
plt.show()

**Key insight.** For every conservation metric the pathogenic
distribution is shifted rightward relative to benign — as expected
from basic evolutionary principles. phyloP100way_vertebrate is the
most discriminative (and ranks #1 in mean|SHAP| — see notebook 06).

## AA chemistry features

For every missense `ref_aa → alt_aa` we compute physicochemical
differences: hydrophobicity, molecular weight, pI, volume, polarity,
charge. Plus the two canonical substitution-severity scores: BLOSUM62
and Grantham.

In [ ]:
chem_cols = ["BLOSUM62_score", "Grantham_distance",
             "polarity_change", "volume_change", "charge_change"]
fig, axes = plt.subplots(1, len(chem_cols), figsize=(15, 3.5))
for ax, col in zip(axes, chem_cols):
    for lbl, color in [(0, "#2ecc71"), (1, "#e74c3c")]:
        ax.hist(train.loc[train["label"] == lbl, col].dropna(),
                bins=30, alpha=0.55, color=color)
    ax.set_title(col)
fig.suptitle("AA chemistry features (pathogenic red, benign green)",
             y=1.05, fontweight="bold")
fig.tight_layout()
plt.show()

## Gene-level constraint (Phase 2 step 1)

Added 2026-04-20. These features describe *the gene itself*, not the
variant — "how intolerant is this gene to any damage?" A variant in a
highly constrained gene has a much higher prior of being pathogenic
than the same variant in a tolerant gene.

| Feature | Source | Intuition |
|---|---|---|
| `pLI` | gnomAD v2.1.1 | Probability the gene is Loss-of-Function intolerant (0–1). |
| `oe_lof_upper` (LOEUF) | gnomAD v2.1.1 | Upper CI of observed/expected LoF. Lower = more constrained. |
| `mis_z` | gnomAD v2.1.1 | Missense Z-score. Higher = less missense tolerated. |
| `oe_mis_upper` | gnomAD v2.1.1 | Upper CI of observed/expected missense. |
| `lof_z` | gnomAD v2.1.1 | LoF Z-score. |

Missing genes (6.7% of training) get **train-fit medians** + an
`is_imputed_gnomad_constraint` flag. That's the only legal way to
impute without leakage.

In [ ]:
constraint_cols = ["pLI", "oe_lof_upper", "mis_z", "oe_mis_upper", "lof_z"]
fig, axes = plt.subplots(1, len(constraint_cols), figsize=(15, 3.5))
for ax, col in zip(axes, constraint_cols):
    for lbl, color in [(0, "#2ecc71"), (1, "#e74c3c")]:
        ax.hist(train.loc[train["label"] == lbl, col].dropna(),
                bins=30, alpha=0.55, color=color)
    ax.set_title(col)
fig.suptitle("gnomAD gene-level constraint features (pathogenic red, benign green)",
             y=1.05, fontweight="bold")
fig.tight_layout()
plt.show()

imputed_rate = train["is_imputed_gnomad_constraint"].mean()
print(f"Imputed constraint rows in training: {imputed_rate:.1%} ({int(imputed_rate*len(train)):,})")

## Population frequency features (gnomAD)

| Feature | Meaning |
|---|---|
| `AF_popmax` | Maximum allele frequency across gnomAD populations. Common → benign (definitional circularity — handled in leakage chapter). |
| `AN` | Allele number (coverage). |
| `AC` | Allele count. |
| `log_AF` | log-transform for long-tailed gnomAD frequencies. |

## Why we *excluded* some features

| Excluded | Reason | See |
|---|---|---|
| `is_common` | 100% benign when True → definitional label leak. | CHANGELOG "Feature hygiene" |
| `chr` one-hot | 15% gain from dataset-composition artifacts, not biology. | CHANGELOG "Feature hygiene" |
| Raw `ref` / `alt` nucleotides | Redundant with `ref_aa` / `alt_aa`. | `src/training.py::select_feature_columns` |
| `gene` identity | Grouping column for the split — including it would let the model memorize train-set gene→label mappings. | `src/training.py::select_feature_columns` |
| `pos` | Coordinate has no biological meaning without gene context. | Same |
| `review_stars` | ClinVar curation quality signal → indirect label leak. | Same |

The test at `tests/test_verify_no_leakage.py` enforces this
exclusion list on every CI run.

---

## Reproduce

```bash
python -m src.dbnsfp_extraction   # conservation + AA chemistry
python -m src.gnomad_extraction   # population frequency
python -m src.gnomad_constraint   # gene-level constraint (Phase 2.1)
python -m src.data_merge
python -m src.feature_analysis    # missense filter + feature selection
```